# ASCII Maze RL Workshop

Train a small LLM to solve ASCII mazes using GRPO (Group Relative Policy Optimization).

**Your task:** Design a reward function that teaches the model to solve mazes.

The model has already been fine-tuned (SFT) to understand the maze format and output
move sequences. Your reward function will guide GRPO to improve its maze-solving accuracy.

## Pipeline
1. **SFT** (done for you) → model knows the format, solves ~68% of 4×4 mazes
2. **GRPO** (your exercise) → reward function refines accuracy to 80%+

## Setup

Run this notebook on Google Colab with a **T4 GPU** runtime.

In [ ]:
# Install deps
!pip install -q trl peft datasets accelerate bitsandbytes torchao 2>&1 | tail -1

import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('torchao').setLevel(logging.ERROR)

import torch
if not torch.cuda.is_available():
    print("⚠️  CUDA not available! Restart runtime: Runtime → Restart session")
else:
    print(f"✓ CUDA OK: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone the maze infrastructure
!git clone -q -b claude-trl https://github.com/StephenJHardy/ascii-maze-rl.git
import sys
sys.path.insert(0, 'ascii-maze-rl')

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.mem_get_info(0)[1] / 1e9:.1f} GB")

## 1. Explore the Maze Format

Let's see what mazes look like and how solutions work.

In [ ]:
from src.maze_gen import generate
from src.maze_repr import to_str, to_prompt, SYSTEM_PROMPT, solution_to_str
from src.maze_verify import extract_moves, simulate, reached_exit, manhattan_progress
from src.reward import compute_reward

# Generate a sample maze
maze = generate(4, 4, seed=42)
print("Maze (4×4):")
print(to_str(maze))
print(f"\nSolution: {solution_to_str(maze.solution_moves)}")
print(f"Solution length: {len(maze.solution_moves)} moves")
print(f"\nEntry: {maze.entry}, Exit: {maze.exit}")

In [ ]:
# See how the prompt looks (this is what the model receives)
print("System prompt:")
print(SYSTEM_PROMPT)
print("\nUser message (just the maze grid):")
print(to_str(maze))

In [ ]:
# See how move parsing and simulation work
test_output = "r r d d d r"
moves = extract_moves(test_output)
print(f"Parsed moves: {moves}")

path = simulate(moves, maze)
print(f"Path through maze: {path}")
print(f"Reached exit: {reached_exit(path, maze)}")
print(f"Valid steps: {len(path) - 1} / {len(moves)}")
print(f"Progress toward exit: {manhattan_progress(path[-1], maze.exit, maze.entry):.2f}")

## 2. Load Model & Baseline

The pre-trained SFT model solves:
- 3×3: 80%
- 4×4: 30%
- 5×5: 0%
- 6×6: 5%

Your reward function will guide GRPO to improve these numbers.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.maze_dataset import MazeDataset, DatasetConfig, SizeConfig, MazeRecord
from src.maze_repr import SYSTEM_PROMPT, to_str

model_id = "StephenJHardy/maze-cuda-sft-5000-qwen2.5-0.5b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto",
)
print(f"Loaded: {model_id}")
print("\nPre-GRPO baseline:")
print("  3x3: 80%, 4x4: 30%, 5x5: 0%, 6x6: 5%")

## 3. Design Your Reward Function ⭐

**This is the main exercise.** Design a reward function that gives
GRPO useful signal to learn from.

We've pre-generated rollouts (8 attempts at each maze, at multiple
temperatures). You can re-score them instantly with different reward
functions — no GPU needed for this exploration phase.

### The challenge
GRPO compares rollouts within a group. If all rollouts score the same,
advantages are zero and GRPO is blind. Your reward must create **variance**
between rollouts while being **informative** (better paths score higher).

### Available tools
- `extract_moves(text)` → parse output to list of moves, or None
- `simulate(moves, maze)` → walk through maze, returns path
- `reached_exit(path, maze)` → did it solve?
- `manhattan_progress(pos, target, origin)` → distance progress (0–1)
- `solve(width, height, walls, start, end)` → BFS shortest path

In [ ]:
from src.maze_gen import Maze, solve as bfs_solve
from src.maze_verify import extract_moves, simulate, reached_exit, manhattan_progress


def reconstruct_maze(walls, entry, exit_, width, height, solution_path):
    frozen_walls = frozenset(frozenset(tuple(c) for c in w) for w in walls)
    return Maze(width=width, height=height, walls=frozen_walls,
               entry=tuple(entry), exit=tuple(exit_),
               solution=tuple(tuple(p) for p in solution_path), seed=0)


def maze_reward(completions, **kwargs):
    """
    Your reward function. Scores each completion.
    Start simple (v1), then iterate based on the analysis below.
    """
    rewards = []
    for i, completion in enumerate(completions):
        if isinstance(completion, list):
            completion = completion[-1]["content"] if completion else ""
        elif isinstance(completion, dict):
            completion = completion.get("content", "")

        maze = reconstruct_maze(
            walls=kwargs["walls"][i], entry=kwargs["entry"][i],
            exit_=kwargs["exit"][i], width=kwargs["width"][i],
            height=kwargs["height"][i], solution_path=kwargs["solution_path"][i],
        )

        # --- v1: naive binary reward (start here) ---
        moves = extract_moves(completion)
        if moves is None:
            reward = 0.0
        else:
            path = simulate(moves, maze)
            reward = 1.0 if reached_exit(path, maze) else 0.0

        rewards.append(reward)
    return rewards

### 3b. Test Your Reward on Pre-generated Rollouts

Re-run this cell after changing your reward function. It's **instant**
(no GPU) — scores pre-saved completions with your current function.

In [ ]:
# Load pre-generated rollouts and score with YOUR reward function
import json as _json, numpy as np

with open('ascii-maze-rl/configs/pregenerated_rollouts.json') as f:
    rollout_data = _json.load(f)

mazes_meta = rollout_data["mazes"]
all_completions = rollout_data["completions"]
temperatures = rollout_data["temperatures"]

print(f"{'Temp':>4s} | {'Mean Reward':>11s} | {'Mean Std':>9s} | "
      f"{'Solved':>8s} | {'Zero-var mazes':>14s}")
print("-" * 60)

for temp in temperatures:
    temp_key = str(temp)
    all_stds = []
    all_rewards = []
    total_solved = 0
    zero_var = 0
    total = len(mazes_meta) * 8

    for maze_idx, meta in enumerate(mazes_meta):
        comps = all_completions[temp_key][maze_idx]
        kwargs = {
            "walls": [meta["walls"]] * 8,
            "entry": [meta["entry"]] * 8,
            "exit": [meta["exit"]] * 8,
            "width": [meta["width"]] * 8,
            "height": [meta["height"]] * 8,
            "solution_path": [meta["solution_path"]] * 8,
        }
        rewards = maze_reward(comps, **kwargs)
        std = np.std(rewards)
        all_stds.append(std)
        all_rewards.append(np.mean(rewards))
        if std < 0.005:
            zero_var += 1
        maze = reconstruct_maze(meta["walls"], meta["entry"], meta["exit"],
                                meta["width"], meta["height"], meta["solution_path"])
        for c in comps:
            moves = extract_moves(c)
            if moves and reached_exit(simulate(moves, maze), maze):
                total_solved += 1

    print(f"{temp:4.1f} | {np.mean(all_rewards):>11.4f} | {np.mean(all_stds):>9.4f} | "
          f"{total_solved:>3d}/{total:<3d} | {zero_var:>6d}/{len(mazes_meta):<6d}")

print("\n💡 Good reward functions have:")
print("   - Mean Std > 0.15 at temp=1.0")
print("   - Few zero-variance mazes")
print("   - If ALL mazes are zero-var, GRPO learns nothing!")

In [ ]:
# Visualize rollouts at a chosen temperature with your reward function
# Change TEMP to explore different temperatures

TEMP = "1.0"  # try: "0.6", "0.8", "1.0", "1.2", "1.4", "1.6", "1.8"

from src.rollout_capture import MazeRollouts, RolloutResult, compute_advantages, save_rollouts
from src.build_rollout_viewer import HTML_TEMPLATE
from dataclasses import asdict
from IPython.display import HTML, display

viz_rollouts = []
for maze_idx, meta in enumerate(mazes_meta):
    comps = all_completions[TEMP][maze_idx]
    maze = reconstruct_maze(meta["walls"], meta["entry"], meta["exit"],
                            meta["width"], meta["height"], meta["solution_path"])
    kwargs = {
        "walls": [meta["walls"]] * 8, "entry": [meta["entry"]] * 8,
        "exit": [meta["exit"]] * 8, "width": [meta["width"]] * 8,
        "height": [meta["height"]] * 8, "solution_path": [meta["solution_path"]] * 8,
    }
    rewards = maze_reward(comps, **kwargs)
    advantages, mean_r, std_r = compute_advantages(rewards)

    rollouts = []
    for j, c in enumerate(comps):
        moves = extract_moves(c)
        path = simulate(moves or [], maze)
        rollouts.append(RolloutResult(
            completion=c, moves_parsed=moves,
            path=[list(p) for p in path], reward=rewards[j],
            solved=reached_exit(path, maze),
            valid_steps=len(path)-1,
            progress=max(manhattan_progress(path[-1], maze.exit, maze.entry), 0.0),
        ))

    viz_rollouts.append(MazeRollouts(
        maze_id=meta["id"], width=meta["width"], height=meta["height"],
        maze_str=meta["maze_str"], entry=meta["entry"], exit=meta["exit"],
        correct_path=meta["solution_path"],
        correct_moves=meta["solution_moves"].split() if isinstance(meta["solution_moves"], str) else meta["solution_moves"],
        solution_length=meta["solution_length"],
        rollouts=rollouts, advantages=advantages,
        reward_mean=mean_r, reward_std=std_r,
    ))

import json as _json
viewer_data = [asdict(r) for r in viz_rollouts]
serialized = _json.dumps(viewer_data).replace('</', '<\\/')
viewer_html = HTML_TEMPLATE.replace('__DATA_PLACEHOLDER__', serialized)

with open('rollout_viewer.html', 'w') as f:
    f.write(viewer_html)
print(f'Rollout viewer (temp={TEMP}): {len(viz_rollouts)} mazes')
print(f'Open rollout_viewer.html in a new tab, or view inline:')

display(HTML(f'<iframe srcdoc="{viewer_html.replace(chr(34), "&quot;")}" '
            f'width="100%" height="600px" style="border:1px solid #333;'
            f'border-radius:4px;"></iframe>'))

### 3c. Iterate

Go back to the reward function cell, improve it, re-run the scoring.

**Progression:**
1. Binary (0/1) → notice high zero-var count
2. Add `-1.0` for unparseable → differentiates gibberish from failures
3. Add partial credit (coverage/progress) → reduces zero-var
4. Use BFS distance instead of manhattan → more accurate progress
5. Add loop penalty → discourages random wandering

Once you're happy with the variance, move to GRPO training below.

## 4. Generate Training Data & Run GRPO

One shot — lock in your reward function and train.
~15-30 min depending on hardware.

In [ ]:
from datasets import Dataset

# Generate GRPO training data
train_config = DatasetConfig(
    name="train",
    algorithm="wilson",
    sizes=[
        SizeConfig(width=3, height=3, count=500, start_seed=0),
        SizeConfig(width=4, height=4, count=1000, start_seed=100_000),
        SizeConfig(width=5, height=5, count=500, start_seed=200_000),
    ],
)
train_ds = MazeDataset.generate(train_config, progress=False)
print(f"Training data: {train_ds.summary()}")

# Convert to HuggingFace format
def maze_records_to_hf(records):
    rows = []
    for r in records:
        rows.append({
            "prompt": [{"role": "system", "content": SYSTEM_PROMPT},
                       {"role": "user", "content": r.maze_str}],
            "walls": r.walls, "entry": r.entry, "exit": r.exit,
            "width": r.width, "height": r.height,
            "solution_path": r.solution_path,
            "solution_moves": r.solution_moves,
            "solution_length": r.solution_length,
        })
    return Dataset.from_list(rows)

train_hf = maze_records_to_hf(train_ds.records)
print(f"HF dataset: {len(train_hf)} mazes")

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig

grpo_config = GRPOConfig(
    loss_type="grpo",              # explicit — TRL defaults to "dapo" now
    output_dir="./maze-grpo-output",
    num_train_epochs=1,
    max_steps=2000,
    per_device_train_batch_size=8,
    num_generations=8,
    max_completion_length=40,
    learning_rate=1e-5,
    beta=0.04,
    logging_steps=100,
    save_steps=1000,
    temperature=1.0,
    report_to="none",
    bf16=False,
    fp16=True,
    gradient_accumulation_steps=1,
)

peft_config = LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[maze_reward],
    args=grpo_config,
    train_dataset=train_hf,
    peft_config=peft_config,
)
print(f"GRPO ready: {grpo_config.max_steps} steps, {grpo_config.num_generations} gen")

In [ ]:
trainer.train()

## 5. Evaluate

Compare your results against the baseline.

In [ ]:
from src.maze_verify import extract_moves, simulate, reached_exit

def generate_solution(mdl, tok, maze, max_new_tokens=40):
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
               {"role": "user", "content": to_str(maze)}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(**inputs, max_new_tokens=max_new_tokens,
                           do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

# Eval data
eval_config = DatasetConfig(name="eval", algorithm="wilson", sizes=[
    SizeConfig(width=3, height=3, count=20, start_seed=900_000),
    SizeConfig(width=4, height=4, count=40, start_seed=910_000),
    SizeConfig(width=5, height=5, count=40, start_seed=920_000),
    SizeConfig(width=6, height=6, count=40, start_seed=930_000),
])
eval_ds = MazeDataset.generate(eval_config, progress=False)

trained_model = trainer.model
results = {}
for record in eval_ds:
    maze = record.to_maze()
    response = generate_solution(trained_model, tokenizer, maze)
    moves = extract_moves(response)
    path = simulate(moves or [], maze)
    size = f"{record.width}x{record.height}"
    if size not in results:
        results[size] = {"total": 0, "solved": 0}
    results[size]["total"] += 1
    results[size]["solved"] += int(reached_exit(path, maze))

pre = {"3x3": 80.0, "4x4": 30.0, "5x5": 0.0, "6x6": 5.0}
print("Results comparison:")
print(f"{'Size':>5s}  {'Before':>8s}  {'After':>8s}  {'Change':>8s}")
print("-" * 35)
for size in sorted(results):
    post_rate = results[size]['solved'] / results[size]['total'] * 100
    pre_rate = pre.get(size, 0.0)
    print(f"{size:>5s}  {pre_rate:7.1f}%  {post_rate:7.1f}%  {post_rate-pre_rate:+7.1f}pp")

## Solution (if stuck)

The best reward function uses BFS distance (actual maze distance to exit),
not manhattan distance. It also penalizes loops and gibberish.

In [ ]:
# =================================================================
# SOLUTION: BFS-distance reward function
# =================================================================
# Uncomment and use this if you get stuck. But try designing your
# own first — the learning is in the iteration!
#
# def maze_reward(completions, **kwargs):
#     from src.maze_gen import solve as bfs_solve
#     rewards = []
#     for i, completion in enumerate(completions):
#         if isinstance(completion, list):
#             completion = completion[-1]["content"] if completion else ""
#         elif isinstance(completion, dict):
#             completion = completion.get("content", "")
#
#         maze = reconstruct_maze(
#             walls=kwargs["walls"][i], entry=kwargs["entry"][i],
#             exit_=kwargs["exit"][i], width=kwargs["width"][i],
#             height=kwargs["height"][i], solution_path=kwargs["solution_path"][i],
#         )
#
#         moves = extract_moves(completion)
#
#         # Unparseable = negative (forces model away from gibberish)
#         if moves is None:
#             rewards.append(-1.0)
#             continue
#
#         path = simulate(moves, maze)
#         valid_steps = len(path) - 1
#         optimal_len = len(maze.solution) - 1
#
#         # Solved! Reward 0.6-1.0 with efficiency bonus
#         if reached_exit(path, maze):
#             efficiency = optimal_len / max(len(moves), optimal_len)
#             rewards.append(0.6 + 0.4 * efficiency)
#             continue
#
#         # Partial credit: BFS distance from final position to exit
#         # This measures ACTUAL maze distance, not as-the-crow-flies
#         final_pos = path[-1]
#         remaining = bfs_solve(maze.width, maze.height, maze.walls, final_pos, maze.exit)
#         remaining_dist = len(remaining) - 1 if remaining else optimal_len
#         bfs_progress = max(1.0 - remaining_dist / max(optimal_len, 1), 0.0)
#
#         # Loop penalty: penalize revisiting cells
#         unique_cells = len(set(path))
#         loop_penalty = 1.0 - (len(path) - unique_cells) / max(len(path), 1)
#
#         # Combine: BFS progress (70%) + loop penalty (20%) + coverage (10%)
#         coverage = min(valid_steps / max(optimal_len, 1), 1.0)
#         partial = 0.5 * (0.7 * bfs_progress + 0.2 * loop_penalty + 0.1 * coverage)
#         rewards.append(partial)
#
#     return rewards
#
# Key design principles:
#   - BFS distance: measures actual maze distance, not manhattan (ignores walls)
#   - Loop penalty: discourages random wandering that revisits cells
#   - Smooth gradient: every step closer to exit increases reward
#   - Clear gap: solved (0.6+) vs partial (0-0.5) vs gibberish (-1.0)
